# Chunk Large Files and Loop Over Chunks in LLM Calls

In [1]:
from openai import AzureOpenAI
from dotenv import load_dotenv
import os
import pandas as pd
from langchain.docstore.document import Document as LangchainDocument
from langchain.text_splitter import RecursiveCharacterTextSplitter
import tiktoken
from docx import Document
from PyPDF2 import PdfReader

# langchain-community
# langchain-openai
load_dotenv()

True

In [2]:
azure_endpoint = os.getenv("AZURE_ENDPOINT")
api_key = os.getenv("API_KEY")
api_version = os.getenv("API_VERSION")
deployment_name = os.getenv("DEPLOYMENT_NAME")

# Count Tokens

In [3]:
def get_token_count(text):

    encoding = tiktoken.encoding_for_model("gpt-4o")
    num_tokens = len(encoding.encode(text))
    return num_tokens

text = "This is an example sentence to count tokens for GPT-4o."
get_token_count(text)

14

# Load Docs

In [4]:
client = AzureOpenAI(
    azure_endpoint = azure_endpoint,
    api_key = api_key,
    api_version = api_version
)

In [5]:
# # Excel
# def extract_text_from_excel(file):
#     xls = pd.ExcelFile(file)
#     all_sheets = []
#     for sheet_name in xls.sheet_names:
#         df = xls.parse(sheet_name)
#         all_sheets.append(f"--- Sheet: {sheet_name} ---\n{df.to_string(index=False)}")
#     return "\n\n".join(all_sheets)

# excel_path = "/Users/jordan.bakerman/Library/CloudStorage/OneDrive-Ralliant/Desktop/PSR_Unlock_Growth/employee_comments_analysis.xlsx"
# docs = extract_text_from_excel(excel_path)

In [6]:
# # CSV
# def extract_text_from_csv(file):
#     df = pd.read_csv(file)
#     return df.to_string(index=False)

# csv_path = "/Users/jordan.bakerman/Library/CloudStorage/OneDrive-Ralliant/Desktop/PSR_Unlock_Growth/employee_comments_analysis copy.csv"
# docs = extract_text_from_csv(csv_path)

In [7]:
# # Word
# def extract_text_from_docx(file):
#     doc = Document(file)
#     return "\n".join([para.text for para in doc.paragraphs])

# word_path = "/Users/jordan.bakerman/Library/CloudStorage/OneDrive-Ralliant/Desktop/Stuff/The Project Gutenberg eBook of The Wonderful Wizard of Oz.docx"
# docs = extract_text_from_docx(word_path)

In [8]:
# # TXT
# def extract_text_from_txt(file):
#     return file.read().decode("utf-8")

# txt_path = "/Users/jordan.bakerman/Library/CloudStorage/OneDrive-Ralliant/Desktop/Stuff/The Project Gutenberg eBook of The Wonderful Wizard of Oz.txt"
# docs = extract_text_from_txt(txt_path)

In [9]:
# PDF
def extract_text_from_pdf(file):
    reader = PdfReader(file)
    return "\n".join([page.extract_text() or "" for page in reader.pages])

pdf_path = "/Users/jordan.bakerman/Library/CloudStorage/OneDrive-Ralliant/Desktop/Stuff/wizard_of_oz.pdf"
docs = extract_text_from_pdf(pdf_path)
get_token_count(docs)

54616

# LLM with LangChain

In [10]:
def strip_text(text):
    temp = text.split()
    return " ".join(temp)

In [11]:
def chunk_text(text, chunk_size=20000, chunk_overlap=200):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunked_docs = text_splitter.split_documents([LangchainDocument(page_content=text)])
    return chunked_docs

In [12]:
def summary_chain(split_docs):

    summaries = []
    for i in range(len(split_docs)):
        try:
            print(f"Summarizing chunk {i+1}/{len(split_docs)}...")
            
            instructions = "Summarize the following text."
            messages = [
                {"role": "system", "content": instructions},
                {"role": "user", "content": split_docs[i].page_content}
            ]

            completion = client.chat.completions.create(
                model=deployment_name,
                messages=messages,
                max_tokens=5000,
                temperature=0
            )

            summaries.append(completion.choices[0].message.content)

        except Exception as e:
            #print(f"⚠️ Chunk {i+1} failed due to policy violation or other error.")
            print(f"Error: {e}")
            continue
        
    return summaries

In [13]:
def final_summary(summaries):

    combined_summary_text = "\n\n".join(summaries)

    instructions = "Summarize the following text."
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": combined_summary_text}
    ]

    completion = client.chat.completions.create(
        model=deployment_name,
        messages=messages,
        max_tokens=5000,
        temperature=0
    )

    return completion.choices[0].message.content

In [14]:
pdf_path = "/Users/jordan.bakerman/Library/CloudStorage/OneDrive-Ralliant/Desktop/Stuff/wizard_of_oz.pdf"
docs = extract_text_from_pdf(pdf_path)

docs_clean = strip_text(docs)

split_docs = chunk_text(docs_clean)

summaries = summary_chain(split_docs)

total_summary = final_summary(summaries)

print("\n✅ Final Summary:\n", total_summary)

Summarizing chunk 1/12...
Summarizing chunk 2/12...
Summarizing chunk 3/12...
Summarizing chunk 4/12...
Summarizing chunk 5/12...
Summarizing chunk 6/12...
Summarizing chunk 7/12...
Summarizing chunk 8/12...
Summarizing chunk 9/12...
Summarizing chunk 10/12...
Summarizing chunk 11/12...
Summarizing chunk 12/12...

✅ Final Summary:
 *The Wonderful Wizard of Oz* by L. Frank Baum, available through Project Gutenberg, is a modern fairy tale written in 1900 to entertain children without the grim elements of traditional folklore. The story follows Dorothy, an orphan living in Kansas, who is swept by a cyclone to the magical Land of Oz. Her house accidentally kills the Wicked Witch of the East, freeing the Munchkins. Dorothy, wearing the Witch's silver shoes, is advised to follow the yellow brick road to the Emerald City to seek help from the Wizard of Oz. Along the way, she befriends the Scarecrow, Tin Woodman, and Cowardly Lion, who join her to seek brains, a heart, and courage, respectivel

In [15]:
# Get summary of entire document without chunking

def summarize_full_text(text):

    instructions = "Summarize the following text."
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": text}
    ]

    completion = client.chat.completions.create(
        model=deployment_name,
        messages=messages,
        max_tokens=5000,
        temperature=0
    )

    return completion.choices[0].message.content

full_text_summary = summarize_full_text(docs_clean)
print("\n✅ Full Text Summary:\n", full_text_summary)


✅ Full Text Summary:
 "The Wonderful Wizard of Oz" by L. Frank Baum is a classic tale about Dorothy, a young girl from Kansas, who is swept away by a cyclone to the magical Land of Oz. Accompanied by her dog Toto, she embarks on a journey to meet the Wizard of Oz, hoping he can help her return home. Along the way, she befriends the Scarecrow, who desires brains; the Tin Woodman, who seeks a heart; and the Cowardly Lion, who wants courage. Together, they face challenges, including encounters with the Wicked Witch of the West and other obstacles. Ultimately, Dorothy learns that her silver shoes have the power to take her home, and her companions find that they already possess the qualities they were seeking. The story emphasizes themes of self-discovery, friendship, and the idea that the answers to life's challenges often lie within ourselves.
